# Evaluate Qwen3-0.6B in thinking mode

This sampled benchmark preserves the strict one-call protocol while enabling Qwen3 thinking with its recommended decoding settings. It runs three fixed seeds independently, guards against unbounded thinking, and keeps each seed's remote artifacts isolated under `/content/outputs/evaluations/thinking/<seed>/` for preservation under `outputs/evaluations/thinking/<seed>/` locally.

Primary metrics:
- final-answer and semantic task accuracy
- tool-call validity
- generated-token usage per turn
- thinking-budget exhaustion rate
- per-seed and cross-seed aggregates

The deterministic non-thinking benchmark remains in `notebooks/evaluate_qwen3_calculator.ipynb`.

In [ ]:
# @title Install runtime dependencies
%pip install -q "transformers==5.9.0" accelerate tqdm

In [ ]:
# @title Configuration and dataset loading
from __future__ import annotations

import hashlib
import json
import re
from importlib.metadata import version
from pathlib import Path
from statistics import mean
from typing import Any

import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_PATH = Path("/content/data/calculator_qwen3")
SPLIT = "test"
MAX_EXAMPLES = None  # Set an integer for a quick smoke test.
SEEDS = (20260719, 20260720, 20260721)
BATCH_SIZE = 16
MAX_TOOL_ROUNDS = 6
MAX_TOOL_CALLS = 6
MAX_NEW_TOKENS = 512
ENABLE_THINKING = True
DO_SAMPLE = True
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
RESUME = False  # Sampled runs require RNG-state restoration before resume is safe.
EVALUATOR_VERSION = 1
RESULTS_ROOT = Path("/content/outputs/evaluations/thinking")

if not DATASET_PATH.exists():
    local_path = Path("data/calculator_qwen3")
    if local_path.exists():
        DATASET_PATH = local_path
    else:
        raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

from datasets import load_from_disk

def remove_null_fields(value):
    if isinstance(value, dict):
        return {key: remove_null_fields(child) for key, child in value.items() if child is not None}
    if isinstance(value, list):
        return [remove_null_fields(child) for child in value]
    return value

hf_dataset = load_from_disk(str(DATASET_PATH))
dataset_fingerprints = {name: split._fingerprint for name, split in hf_dataset.items()}
DATASET_SHA256 = hashlib.sha256(
    json.dumps(dataset_fingerprints, sort_keys=True).encode("utf-8")
).hexdigest()
all_records_count = sum(len(split) for split in hf_dataset.values())
records = [remove_null_fields(dict(row)) for row in hf_dataset[SPLIT]]
if MAX_EXAMPLES is not None:
    records = records[:MAX_EXAMPLES]

assert all_records_count == 1_000, f"Expected 1,000 rows, found {all_records_count}"
assert records, f"No rows found for split={SPLIT!r}"
SELECTED_RECORDS_SHA256 = hashlib.sha256(
    json.dumps(records, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
print(f"Loaded {all_records_count:,} rows; evaluating {len(records):,} {SPLIT} rows")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# @title Load Qwen3-0.6B for batched inference
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # Required for decoder-only batched generation.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
MODEL_REVISION = (
    getattr(model.config, "_commit_hash", None)
    or tokenizer.init_kwargs.get("_commit_hash")
    or "unresolved"
)
DEVICE_NAME = (
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
)
RUNTIME_METADATA_BASE = {
    "model_revision": MODEL_REVISION,
    "transformers_version": transformers.__version__,
    "accelerate_version": version("accelerate"),
    "torch_version": torch.__version__,
    "dtype": str(next(model.parameters()).dtype),
    "device": DEVICE_NAME,
    "dataset_sha256": DATASET_SHA256,
    "selected_records_sha256": SELECTED_RECORDS_SHA256,
    "evaluator_version": EVALUATOR_VERSION,
}


def make_run_config(seed: int) -> dict[str, Any]:
    return {
        "model": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "dataset_sha256": DATASET_SHA256,
        "selected_records_sha256": SELECTED_RECORDS_SHA256,
        "split": SPLIT,
        "max_examples": MAX_EXAMPLES,
        "num_examples": len(records),
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_tool_rounds": MAX_TOOL_ROUNDS,
        "max_tool_calls": MAX_TOOL_CALLS,
        "enable_thinking": ENABLE_THINKING,
        "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "use_cache": True,
        "stop_strings": ["</tool_call>"],
        "padding_side": tokenizer.padding_side,
        "transformers_version": transformers.__version__,
        "torch_version": torch.__version__,
        "dtype": str(next(model.parameters()).dtype),
        "device": DEVICE_NAME,
        "seed": seed,
        "evaluator_version": EVALUATOR_VERSION,
    }


def make_run_signature(seed: int) -> str:
    return hashlib.sha256(
        json.dumps(make_run_config(seed), sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]


print(
    f"Loaded {MODEL_ID}@{MODEL_REVISION} with dtype={DTYPE}, "
    f"batch_size={BATCH_SIZE}, thinking={ENABLE_THINKING}"
)

In [ ]:
# @title Tool execution and response parsing
TOOL_TAG_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)
ANSWER_PATTERNS = (
    re.compile(
        r"(?:the\s+)?answer\s+is\s*[:=]?\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?:the\s+)?(?:final\s+)?result(?:\s+of.*?)?\s+is\s*[:=]?\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE | re.DOTALL,
    ),
    re.compile(r"\\boxed\{\s*(-?\d+)\s*\}"),
    re.compile(
        r"final(?:\s+answer)?\s*[:=]\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE,
    ),
    re.compile(r"^\s*\$?\s*(-?\d+)\s*\$?\s*\.?\s*$"),
)


def calculator(op: str, a: int, b: int) -> int:
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    raise ValueError(f"Unsupported calculator operation: {op}")


def validate_semantic_trace(
    expression: str,
    calls: list[dict[str, Any]],
    final_answer: int,
) -> tuple[bool, bool]:
    """Accept any legal precedence-respecting reduction order."""
    pieces = re.findall(r"\d+|[+*-]", expression)
    initial = tuple(int(piece) if piece.isdigit() else piece for piece in pieces)
    possible_states: set[tuple[int | str, ...]] = {initial}

    for call in calls:
        op, a, b = call["op"], call["a"], call["b"]
        next_states: set[tuple[int | str, ...]] = set()
        for tokens in possible_states:
            multiplication_remains = "*" in tokens
            for index in range(1, len(tokens), 2):
                if tokens[index] != op:
                    continue
                # Multiplications may be reduced in any independent order. Once
                # they are gone, addition/subtraction must remain left-associative.
                if op in {"+", "-"} and (multiplication_remains or index != 1):
                    continue
                if tokens[index - 1] != a or tokens[index + 1] != b:
                    continue
                result = calculator(op, a, b)
                reduced = tokens[: index - 1] + (result,) + tokens[index + 2 :]
                next_states.add(reduced)
        if not next_states:
            return False, False
        possible_states = next_states

    complete = any(
        len(tokens) == 1 and tokens[0] == final_answer
        for tokens in possible_states
    )
    return bool(calls), complete


def parse_tool_call(text: str) -> tuple[dict[str, Any] | None, str | None]:
    matches = TOOL_TAG_RE.findall(text)
    if not matches:
        if "<tool_call>" in text:
            return None, "unclosed_tool_call"
        return None, None
    if len(matches) != 1:
        return None, "multiple_tool_calls_in_one_turn"
    try:
        payload = json.loads(matches[0])
        function_name = payload["name"]
        arguments = payload["arguments"]
        if function_name != "calculator":
            return None, "wrong_tool_name"
        if set(arguments) != {"op", "a", "b"}:
            return None, "wrong_argument_schema"
        if arguments["op"] not in {"+", "-", "*"}:
            return None, "invalid_operator"
        if type(arguments["a"]) is not int or type(arguments["b"]) is not int:
            return None, "non_integer_operand"
        return {"name": function_name, "arguments": arguments}, None
    except (KeyError, TypeError, json.JSONDecodeError):
        return None, "malformed_tool_call"


def parse_final_answer(text: str) -> int | None:
    for pattern in ANSWER_PATTERNS:
        matches = pattern.findall(text)
        if matches:
            return int(matches[-1])
    return None


def render_prompt(messages: list[dict[str, Any]], tools: list[dict[str, Any]]) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=ENABLE_THINKING,
    )


def generate_turns(prompts: list[str]) -> list[dict[str, Any]]:
    """Generate sampled turns and retain per-sequence token-budget metadata."""
    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=False,
        add_special_tokens=False,
        return_tensors="pt",
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    padded_input_width = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            use_cache=True,
            stop_strings=["</tool_call>"],
            tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[:, padded_input_width:]
    texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
    turn_outputs: list[dict[str, Any]] = []
    for token_row, text in zip(generated, texts):
        token_ids = token_row.tolist()
        complete_tool_call = "</tool_call>" in text
        try:
            first_eos = token_ids.index(tokenizer.eos_token_id)
            # Stop-string completions are padded with EOS; natural completions
            # generate EOS and include it in their token usage.
            generated_token_count = first_eos if complete_tool_call else first_eos + 1
        except ValueError:
            generated_token_count = len(token_ids)
        budget_exhausted = (
            generated_token_count >= MAX_NEW_TOKENS and not complete_tool_call
        )
        turn_outputs.append(
            {
                "text": text,
                "generated_token_count": generated_token_count,
                "thinking_budget_exhausted": budget_exhausted,
            }
        )
    return turn_outputs


def expected_trace(record: dict[str, Any]) -> list[dict[str, Any]]:
    return [
        message["tool_calls"][0]["function"]["arguments"]
        for message in record["messages"]
        if message["role"] == "assistant" and "tool_calls" in message
    ]

In [ ]:
# @title Batched agent state and per-example scoring
import time


def new_state(
    record: dict[str, Any], seed: int, run_signature: str
) -> dict[str, Any]:
    return {
        "record": record,
        "seed": seed,
        "run_signature": run_signature,
        "messages": [dict(message) for message in record["messages"][:2]],
        "tools": record["tools"],
        "expected_calls": expected_trace(record),
        "model_calls": [],
        "generated_turns": [],
        "generated_token_counts": [],
        "thinking_budget_exhausted": False,
        "predicted_answer": None,
        "error": None,
        "tool_calls_valid": True,
        "done": False,
        "started": time.perf_counter(),
    }


def process_turn(
    state: dict[str, Any], turn_output: dict[str, Any], turn_index: int
) -> None:
    text = turn_output["text"].strip()
    generated_token_count = turn_output["generated_token_count"]
    budget_exhausted = turn_output["thinking_budget_exhausted"]

    # Enforce one tool call -> result -> next generation.
    closing_tag = "</tool_call>"
    first_call_end = text.find(closing_tag)
    if first_call_end >= 0:
        text = text[: first_call_end + len(closing_tag)]

    state["generated_turns"].append(text)
    state["generated_token_counts"].append(generated_token_count)
    if budget_exhausted and first_call_end < 0:
        state["thinking_budget_exhausted"] = True
        state["error"] = "thinking_budget_exhausted"
        state["done"] = True
        return

    parsed_call, parse_error = parse_tool_call(text)

    if parse_error is not None:
        state["error"] = parse_error
        state["tool_calls_valid"] = False
        state["done"] = True
        return
    if parsed_call is None:
        state["predicted_answer"] = parse_final_answer(text)
        if state["predicted_answer"] is None:
            state["error"] = "missing_final_answer"
        state["done"] = True
        return
    if len(state["model_calls"]) >= MAX_TOOL_CALLS:
        state["error"] = "max_tool_calls_exceeded"
        state["done"] = True
        return

    arguments = parsed_call["arguments"]
    try:
        result = calculator(arguments["op"], arguments["a"], arguments["b"])
    except ValueError:
        state["error"] = "calculator_rejected_call"
        state["tool_calls_valid"] = False
        state["done"] = True
        return

    call_id = f"eval_call_{turn_index:02d}"
    state["messages"].append(
        {
            "role": "assistant",
            "content": text.split("<tool_call>", 1)[0].strip(),
            "tool_calls": [
                {
                    "id": call_id,
                    "type": "function",
                    "function": {"name": "calculator", "arguments": arguments},
                }
            ],
        }
    )
    state["messages"].append(
        {
            "role": "tool",
            "name": "calculator",
            "tool_call_id": call_id,
            "content": json.dumps({"result": result}),
        }
    )
    state["model_calls"].append(arguments)


def finalize_state(state: dict[str, Any]) -> dict[str, Any]:
    record = state["record"]
    final_answer = record["metadata"]["final_answer"]
    predicted_answer = state["predicted_answer"]
    model_calls = state["model_calls"]
    expected_calls = state["expected_calls"]
    answer_correct = predicted_answer == final_answer
    exact_trace = model_calls == expected_calls
    valid_tool_use = bool(model_calls) and state["tool_calls_valid"]
    parseable_completion = predicted_answer is not None
    semantic_trace_valid, semantic_trace_complete = validate_semantic_trace(
        record["expression"], model_calls, final_answer
    )
    task_success = (
        answer_correct
        and valid_tool_use
        and semantic_trace_complete
    )
    generated_token_counts = state["generated_token_counts"]
    return {
        "id": record["id"],
        "seed": state["seed"],
        "run_signature": state["run_signature"],
        "expression": record["expression"],
        "tier": record["metadata"]["tier"],
        "final_answer": final_answer,
        "predicted_answer": predicted_answer,
        "answer_correct": answer_correct,
        "used_tool": bool(model_calls),
        "valid_tool_use": valid_tool_use,
        "parseable_completion": parseable_completion,
        "tool_call_count": len(model_calls),
        "expected_tool_call_count": len(expected_calls),
        "tool_call_count_correct": len(model_calls) == len(expected_calls),
        "semantic_trace_valid": semantic_trace_valid,
        "semantic_trace_complete": semantic_trace_complete,
        "exact_trace": exact_trace,
        "task_success": task_success,
        "reference_trace_success": task_success and exact_trace,
        "error": state["error"],
        "model_calls": model_calls,
        "expected_calls": expected_calls,
        "generated_turns": state["generated_turns"],
        "generated_token_counts": generated_token_counts,
        "total_generated_tokens": sum(generated_token_counts),
        "num_generation_turns": len(generated_token_counts),
        "thinking_budget_exhausted": state["thinking_budget_exhausted"],
        "completion_time_from_start_seconds": round(
            time.perf_counter() - state["started"], 3
        ),
    }

In [ ]:
# @title Run three independent sampled seeds
assert not RESUME, "Sampled runs must not resume without restoring RNG state"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


def run_seed(seed: int) -> dict[str, Any]:
    set_seed(seed)
    run_signature = make_run_signature(seed)
    seed_dir = RESULTS_ROOT / str(seed)
    seed_dir.mkdir(parents=True, exist_ok=True)
    predictions_path = seed_dir / "predictions.jsonl"
    states = [new_state(record, seed, run_signature) for record in records]
    results: list[dict[str, Any]] = []
    evaluation_started = time.perf_counter()

    with predictions_path.open("w", encoding="utf-8") as handle, tqdm(
        total=len(states), desc=f"Thinking seed {seed}"
    ) as progress:
        for turn_index in range(MAX_TOOL_ROUNDS + 1):
            active_states = [state for state in states if not state["done"]]
            if not active_states:
                break

            rendered_prompts = [
                render_prompt(state["messages"], state["tools"])
                for state in active_states
            ]
            tokenized_for_lengths = tokenizer(
                rendered_prompts, add_special_tokens=False
            )["input_ids"]
            prompt_states = [
                (len(token_ids), prompt, state)
                for token_ids, prompt, state in zip(
                    tokenized_for_lengths, rendered_prompts, active_states
                )
            ]
            prompt_states.sort(key=lambda item: item[0])

            for start in range(0, len(prompt_states), BATCH_SIZE):
                batch = prompt_states[start : start + BATCH_SIZE]
                prompts = [prompt for _, prompt, _ in batch]
                batch_states = [state for _, _, state in batch]
                turn_outputs = generate_turns(prompts)

                for state, turn_output in zip(batch_states, turn_outputs):
                    process_turn(state, turn_output, turn_index)
                    if state["done"]:
                        result = finalize_state(state)
                        results.append(result)
                        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
                        handle.flush()
                        progress.update(1)

        unfinished = [state for state in states if not state["done"]]
        for state in unfinished:
            state["error"] = "agent_loop_exhausted"
            state["done"] = True
            result = finalize_state(state)
            results.append(result)
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
            handle.flush()
            progress.update(1)

    wall_time_seconds = time.perf_counter() - evaluation_started
    results.sort(key=lambda row: row["id"])
    print(
        f"Seed {seed}: saved {len(results):,} predictions in "
        f"{wall_time_seconds:.1f}s ({len(results) / wall_time_seconds:.2f} examples/s)"
    )
    return {
        "seed": seed,
        "run_signature": run_signature,
        "results": results,
        "wall_time_seconds": wall_time_seconds,
        "predictions_path": predictions_path,
    }


seed_runs = [run_seed(seed) for seed in SEEDS]

In [ ]:
# @title Save per-seed and aggregate thinking metrics
from collections import Counter

METRIC_FIELDS = (
    "answer_correct",
    "used_tool",
    "valid_tool_use",
    "parseable_completion",
    "tool_call_count_correct",
    "semantic_trace_valid",
    "semantic_trace_complete",
    "exact_trace",
    "task_success",
    "reference_trace_success",
    "thinking_budget_exhausted",
)


def rates(rows: list[dict[str, Any]]) -> dict[str, float]:
    return {
        name: round(mean(float(row[name]) for row in rows), 4)
        for name in METRIC_FIELDS
    }


def percentile(values: list[int], fraction: float) -> int:
    if not values:
        return 0
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, round((len(ordered) - 1) * fraction)))
    return ordered[index]


def summarize_seed(seed_run: dict[str, Any]) -> dict[str, Any]:
    seed = seed_run["seed"]
    rows = seed_run["results"]
    turn_token_counts = [
        count for row in rows for count in row["generated_token_counts"]
    ]
    total_tokens = sum(row["total_generated_tokens"] for row in rows)
    runtime_metadata = {
        **RUNTIME_METADATA_BASE,
        "seed": seed,
        "run_signature": seed_run["run_signature"],
    }
    metrics: dict[str, Any] = {
        "model": MODEL_ID,
        "dataset": str(DATASET_PATH),
        "split": SPLIT,
        "num_examples": len(rows),
        "reproducibility": runtime_metadata,
        "inference_config": make_run_config(seed),
        "overall": rates(rows),
        "token_usage": {
            "total_generated_tokens": total_tokens,
            "mean_generated_tokens_per_example": round(total_tokens / len(rows), 3),
            "total_generation_turns": len(turn_token_counts),
            "mean_generated_tokens_per_turn": round(mean(turn_token_counts), 3),
            "p95_generated_tokens_per_turn": percentile(turn_token_counts, 0.95),
        },
        "by_tier": {},
        "errors": dict(Counter(row["error"] or "none" for row in rows)),
        "wall_time_seconds": round(seed_run["wall_time_seconds"], 3),
        "examples_per_second": round(len(rows) / seed_run["wall_time_seconds"], 4),
    }
    for tier in ("easy", "medium", "hard"):
        tier_rows = [row for row in rows if row["tier"] == tier]
        if tier_rows:
            metrics["by_tier"][tier] = {
                "num_examples": len(tier_rows),
                **rates(tier_rows),
            }

    metrics_path = RESULTS_ROOT / str(seed) / "metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics


per_seed_metrics = [summarize_seed(seed_run) for seed_run in seed_runs]
aggregate_metrics = {
    "model": MODEL_ID,
    "dataset": str(DATASET_PATH),
    "split": SPLIT,
    "seeds": list(SEEDS),
    "num_examples_per_seed": len(records),
    "mean_over_seeds": {
        name: round(mean(metrics["overall"][name] for metrics in per_seed_metrics), 4)
        for name in METRIC_FIELDS
    },
    "mean_token_usage_over_seeds": {
        name: round(mean(metrics["token_usage"][name] for metrics in per_seed_metrics), 3)
        for name in (
            "total_generated_tokens",
            "mean_generated_tokens_per_example",
            "mean_generated_tokens_per_turn",
            "p95_generated_tokens_per_turn",
        )
    },
    "per_seed": [
        {
            "seed": metrics["reproducibility"]["seed"],
            "run_signature": metrics["reproducibility"]["run_signature"],
            "overall": metrics["overall"],
            "token_usage": metrics["token_usage"],
            "wall_time_seconds": metrics["wall_time_seconds"],
        }
        for metrics in per_seed_metrics
    ],
}
aggregate_path = RESULTS_ROOT / "aggregate_metrics.json"
aggregate_path.write_text(json.dumps(aggregate_metrics, indent=2), encoding="utf-8")
print(json.dumps(aggregate_metrics, indent=2))
print(f"Saved thinking benchmark metrics to {RESULTS_ROOT}")

In [ ]:
# @title Inspect representative failures from the first seed
first_seed_results = seed_runs[0]["results"]
failures = [row for row in first_seed_results if not row["task_success"]]
print(f"Seed {SEEDS[0]} task failures: {len(failures)}/{len(first_seed_results)}")
for row in failures[:5]:
    print("\n", "=" * 80)
    print("ID:", row["id"], "Tier:", row["tier"])
    print("Expression:", row["expression"])
    print("Expected answer:", row["final_answer"], "Predicted:", row["predicted_answer"])
    print("Error:", row["error"])
    print("Expected calls:", row["expected_calls"])
    print("Model calls:", row["model_calls"])
    print("Last turn:", row["generated_turns"][-1] if row["generated_turns"] else "<none>")

## Result interpretation

Use the aggregate metrics to compare sampled thinking behavior across seeds, and inspect each seed independently before drawing conclusions. `thinking_budget_exhausted` identifies trajectories that consumed all 512 tokens without completing a tool call; these are not silently retried in non-thinking mode.

`task_success` remains policy-aware correctness, while `exact_trace` only measures equality with the single stored reference trace. The strict one-call protocol is enforced at generation time for every seed.

## Output layout

Each sampled run writes remotely to `/content/outputs/evaluations/` and is preserved locally as:

```text
outputs/evaluations/thinking/<seed>/predictions.jsonl
outputs/evaluations/thinking/<seed>/metrics.json
```

The cross-seed summary is preserved as:

```text
outputs/evaluations/thinking/aggregate_metrics.json
```

Every prediction stores the generated text and token count for each turn, total generated tokens, budget-exhaustion status, tool calls, semantic trace scores, and final-answer scores.